# 1. Instalación de librerías

In [1]:
!pip install -q decord transformers evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.6 MB/s eta 0:00:00


In [2]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from decord import VideoReader, cpu
from tqdm import tqdm
from datasets import Dataset

# 2. Configuración y rutas

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# MODELO
MODEL_CHECKPOINT = "MCG-NJU/videomae-base"

# RUTAS
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/"
CSV_TRAIN_MASTER = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1_master.csv"
CSV_TEST = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"

# 3. Carga de datos

In [5]:
print("Cargando particiones fijas...")
df_train_master = pd.read_csv(CSV_TRAIN_MASTER)
df_test = pd.read_csv(CSV_TEST)

Cargando particiones fijas...


In [6]:
# Limpieza de índices y etiquetas
for df in [df_train_master, df_test]:
    if "Unnamed: 0" in df.columns:
        df.drop(columns=["Unnamed: 0"], inplace=True)
    if "label_task_3_1_merged" in df.columns:
        df.rename(columns={"label_task_3_1_merged": "label"}, inplace=True)
    df["label"] = df["label"].astype(int)

In [7]:
# División Dinámica (90/10) del Train Master
train_df, val_df = train_test_split(
    df_train_master,
    test_size=0.10,
    stratify=df_train_master["label"],
    random_state=42
)

In [8]:
# Convertir las rutas relativas en rutas absolutas completas para decord
def fix_video_paths(df, base_path):
    # Asume que el CSV tiene una columna 'path_video' o 'id_EXIST' + '.mp4'
    # Ajusta esto según el nombre real de tu columna en el CSV de texto
    columna = 'path_video' if 'path_video' in df.columns else 'id_EXIST'

    rutas = []
    for val in df[columna]:
        if not str(val).endswith(".mp4"):
            val = str(val) + ".mp4"

        # IMPORTANTE: A veces el CSV tiene "videos/nombre.mp4".
        ruta_completa = os.path.join(base_path, val) # Ajusta "videos" si es necesario
        rutas.append(ruta_completa)

    df['ruta_absoluta'] = rutas
    return df

In [9]:
train_df = fix_video_paths(train_df.copy(), RUTA_BASE_VIDEOS)
val_df = fix_video_paths(val_df.copy(), RUTA_BASE_VIDEOS)

In [10]:
print(f"\nVídeos en Train: {len(train_df)}")
print(f"Vídeos en Valid: {len(val_df)}")
print("Ejemplo de ruta:", train_df['ruta_absoluta'].iloc[0])


Vídeos en Train: 1805
Vídeos en Valid: 201
Ejemplo de ruta: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/7133053554763287813.mp4


# 4. Función central de extracción

In [11]:
def sample_frame_indices(clip_len, total_frames):
    if total_frames <= clip_len:
        # Si el vídeo es súper corto, repetimos frames
        return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
    else:
        # Muestreo uniforme a lo largo de todo el vídeo
        indices = np.linspace(0, total_frames - 1, num=clip_len, dtype=int)
        return indices.tolist()

# 5. Preparación del dataset de vídeo (pytorch)

In [12]:
from transformers import VideoMAEImageProcessor
from torch.utils.data import Dataset
import decord
from decord import VideoReader, cpu

In [13]:
# Silenciamos los logs de decord para que no ensucien la consola
decord.bridge.set_bridge('torch')

In [14]:
# Cargamos el procesador específico de VideoMAE
print("Cargando VideoMAEImageProcessor...")
processor = VideoMAEImageProcessor.from_pretrained(MODEL_CHECKPOINT)

Cargando VideoMAEImageProcessor...


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

In [15]:
class VideoClassificationDataset(Dataset):
    def __init__(self, df, processor, clip_len=16):
        """
        df: DataFrame que contiene 'ruta_absoluta' y 'label'
        processor: VideoMAEImageProcessor
        clip_len: Número de frames que espera el modelo (VideoMAE usa 16)
        """
        self.rutas = df['ruta_absoluta'].tolist()
        self.etiquetas = df['label'].tolist()
        self.processor = processor
        self.clip_len = clip_len

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, idx):
        ruta_video = self.rutas[idx]
        etiqueta = self.etiquetas[idx]

        # 1. Leemos el vídeo de forma súper eficiente con Decord (solo en CPU para no bloquear la GPU)
        try:
            # ctx=cpu(0) es importante para evitar cuelgues raros en Colab
            vr = VideoReader(ruta_video, ctx=cpu(0))
            total_frames = len(vr)

            # 2. Obtenemos los índices de los 16 frames repartidos uniformemente
            frame_indices = sample_frame_indices(self.clip_len, total_frames)

            # 3. Extraemos SOLO esos 16 frames (devuelve un tensor de PyTorch gracias al bridge)
            # El formato original suele ser [T, H, W, C] (Tiempo, Alto, Ancho, Canales)
            frames = vr.get_batch(frame_indices).numpy()

            # 4. Convertimos la lista de numpy arrays al formato que quiere Hugging Face
            # El procesador espera una lista de arrays numpy o un solo tensor gigante.
            # Le pasamos la lista de frames y él se encarga de redimensionar a 224x224
            inputs = self.processor(list(frames), return_tensors="pt")

            # El procesador devuelve un diccionario {'pixel_values': tensor_gigante}
            # Extraemos el tensor y le quitamos la dimensión extra del 'batch' que pone por defecto
            pixel_values = inputs["pixel_values"][0]

        except Exception as e:
            # A veces un .mp4 puede estar corrupto.
            # Si falla, imprimimos el error y devolvemos un tensor vacío pero con las dimensiones correctas
            # para que el entrenamiento no se corte (es un salvavidas).
            print(f"\n⚠️ Error leyendo {ruta_video}: {e}")
            import torch
            pixel_values = torch.zeros((3, self.clip_len, 224, 224))
            etiqueta = 0 # Valor por defecto

        # Devolvemos el tensor de 4D del vídeo y su etiqueta
        return {"pixel_values": pixel_values, "label": etiqueta}

# 6. Inicialización de los datasets

In [16]:
print("Construyendo los Datasets de PyTorch...")

# Usamos los DataFrames que ya tienen la columna 'ruta_absoluta'
train_dataset = VideoClassificationDataset(train_df, processor, clip_len=16)
valid_dataset = VideoClassificationDataset(val_df, processor, clip_len=16)

Construyendo los Datasets de PyTorch...


In [17]:
# Probamos que funciona correctamente sacando el primer elemento
print("\nComprobando el primer vídeo del dataset (esto extraerá 16 frames)...")
ejemplo = train_dataset[0]
print("Dimensiones del tensor final (Debe ser [Canales(3), Frames(16), Alto(224), Ancho(224)]):")
print(ejemplo["pixel_values"].shape)
print("Etiqueta:", ejemplo["label"])


Comprobando el primer vídeo del dataset (esto extraerá 16 frames)...
Dimensiones del tensor final (Debe ser [Canales(3), Frames(16), Alto(224), Ancho(224)]):
torch.Size([16, 3, 224, 224])
Etiqueta: 0


# 7. Carga de modelo y métricas

In [18]:
from transformers import VideoMAEForVideoClassification
import evaluate
import numpy as np
import torch

print("\nCargando modelo VideoMAEForVideoClassification...")
# Mapeos de etiquetas
id2label = {0: "No Misógino", 1: "Misógino"}
label2id = {"No Misógino": 0, "Misógino": 1}

# Cargamos el modelo pre-entrenado, pero le cambiamos la "cabeza" final
# para que tenga solo 2 neuronas de salida (nuestras 2 clases).
# ignore_mismatched_sizes=True es vital porque el modelo original de HF estaba
# entrenado para 400 clases (Kinetics-400), y nosotros lo forzamos a 2.
model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)


Cargando modelo VideoMAEForVideoClassification...


config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  377MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.v_bias           | UNEXPECTED | 
encoder_to_decoder.weight                                       

In [19]:
# Definimos cómo evaluar (Usaremos F1-Macro y Accuracy, el estándar del TFM)
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)

    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]

    return {"f1": f1, "accuracy": acc}

# 8. Configuración del entrenamiento (Trainer)

In [20]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import os

In [21]:
OUTPUT_DIR = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/VideoMAE_FineTuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

batch_size = 4

In [22]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False, # ¡CRUCIAL PARA VÍDEO! Si es True, HF borra la columna "pixel_values"
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,          # LR típico para fine-tuning de VideoMAE
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=2, # Simulamos un batch de 8 para mejor convergencia
    num_train_epochs=5,          # Le damos 5 épocas (Early Stopping lo parará si es necesario)
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    fp16=True,                   # Aceleración por hardware obligatoria en vídeo
    report_to="none",            # Apagamos wandb/logs externos
    dataloader_num_workers=2,    # Usa 2 procesos en segundo plano para leer los .mp4 más rápido
    dataloader_pin_memory=True
)

In [23]:
# Inicializamos el Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Si no mejora en 2 épocas, para
)

In [24]:
print("\n🚀 Lanzando el entrenamiento de VideoMAE (Ponte cómodo, esto tomará un rato)...")
trainer.train()

print("\n💾 Guardando el modelo definitivo...")
trainer.save_model(os.path.join(OUTPUT_DIR, "modelo_final"))
# También guardamos el procesador para que la inferencia sea fácil luego
processor.save_pretrained(os.path.join(OUTPUT_DIR, "modelo_final"))

print("✅ ¡Entrenamiento completado y guardado con éxito!")


🚀 Lanzando el entrenamiento de VideoMAE (Ponte cómodo, esto tomará un rato)...


Epoch,Training Loss,Validation Loss



⚠️ Error leyendo /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/7136934291056905477.mp4: [16:45:54] /github/workspace/src/video/video_reader.cc:486: Error: av_read_frame failed with 1094995529


RuntimeError: Caught RuntimeError in DataLoader worker process 1.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py", line 89, in default_data_collator
    return torch_default_data_collator(features)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py", line 149, in torch_default_data_collator
    batch[k] = torch.stack([f[k] for f in features])
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: stack expects each tensor to be equal size, but go